# FloodGuard AI: Predictive Hydro-Meteorological Flood Risk Modeling
### 3rd-Semester AI / CSE Academic Project
This notebook trains and exports the Machine Learning classification pipeline (model.pkl) combining the **District Flood Susceptibility Index (DFSI)** with real-time atmospheric triggers (precipitation, humidity, wind, and cloud cover).

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

print('Libraries imported successfully.')

## 1. Load and Clean DFSI Dataset

In [ ]:
df = pd.read_csv('DFSI.csv')
df['District'] = df['District'].astype(str).str.strip().str.replace(r'[\*\r\n"]', '', regex=True)
df['State_Name'] = df['State_Name'].astype(str).str.strip().str.replace(r'[\*\r\n"]', '', regex=True)
df['DFSI'] = pd.to_numeric(df['DFSI'], errors='coerce').fillna(df['DFSI'].dropna().median())

print(f'Total Districts: {len(df)}')
print(f'States / UTs: {df["State_Name"].nunique()}')
df.head(10)

## 2. Generate Multi-Scenario Hydro-Meteorological Training Distribution
Pairing static geomorphological susceptibility (DFSI) with dynamic weather events (monsoon downpours, dry spells, severe storms).

In [ ]:
from train_and_export import generate_synthetic_weather_scenarios

dataset = generate_synthetic_weather_scenarios(df)
print(f'Total scenario observations: {len(dataset)}')
dataset['risk_category'].value_counts()

## 3. Machine Learning Pipeline Training
Using a Standardized Random Forest Classifier with balanced class weights.

In [ ]:
features = ['DFSI', 'precipitation_mm', 'rain_sum_24h', 'humidity_percent', 'temperature_c', 'wind_speed_kmh', 'cloud_cover_percent']
X = dataset[features]
y = dataset['risk_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=180, max_depth=12, class_weight='balanced', random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
print(f'Evaluation Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print('\nClassification Report:\n', classification_report(y_test, y_pred))

## 4. Export Model Artifact
Serializing trained pipeline to model.pkl for Flask API deployment.

In [ ]:
joblib.dump(pipeline, 'model.pkl')
print('Saved model to model.pkl successfully.')